# Industry Project: A Production RAG Assistant over Internal Company Docs (Production-Style)

> **Who is this notebook for?** Someone who knows roughly what an LLM is, has maybe seen a "RAG in 10 lines" demo, and now wonders: *what does a company actually build around it?*
> No API key is needed. Everything runs on a laptop CPU with small open models. Every step is explained in plain English and almost every line of code has a comment.

---

### How this notebook is different from the other RAG notebooks

| `02_rag/01-04` notebooks | This notebook |
|---|---|
| Explain the **building blocks** (embeddings, chunking, vector DBs, reranking, frameworks) | Wires them into **one working system** with tests and metrics |
| Toy sentences, or code that needs an OpenAI key to run | A realistic mini document set **with versions, dates, drafts and contradictions**, fully offline |
| "It retrieved something sensible" | **Measured**: hit-rate@k, MRR, stale-document rate, citation validity, groundedness, refusal rate, latency, cost |
| Show one answer | A reusable `ask()` function with input validation, caching, logging and a refusal path |

---

### The business problem

You work at **Lumora Software** (a fictional 600-person SaaS company that sells a file-sync product called *Lumora Sync*).
The HR, Finance and IT help desks receive about **2,000 internal questions per month** ("How many days can I work from home?", "What's the hotel limit in London?", "VPN error 809?"). Each ticket costs roughly **$6** of staff time, and answers are often inconsistent because **old policy versions are still floating around** the wiki.

**Your job:** build an internal assistant that answers from the official documents, **cites its sources**, **never quotes an outdated policy**, and **says "I don't know"** instead of inventing an answer.

**What success means (agreed with the help-desk leads):**

| KPI | Target |
|---|---|
| Retrieval hit-rate@3 (the right document is in the top 3) | ≥ 90% |
| Answers citing an outdated/draft document | **0** |
| Out-of-scope questions answered with "I don't know" | ≥ 90% |
| Answers with a valid citation | ≥ 90% |
| Latency per answer | < 3 s with a hosted model (our shared laptop CPU is much slower, see Section 15) |

---

### The industry RAG workflow we will follow

```
 OFFLINE (when documents change)                         ONLINE (every question)
 ┌────────────┐  ┌───────────┐  ┌──────────┐  ┌────────┐   ┌────────┐  ┌────────────────┐  ┌────────┐  ┌────────┐
 │ 1 Ingest + │→ │ 2 Chunk   │→ │ 3 Embed  │→ │ 4 Index│   │ Filter │→ │ Hybrid search  │→ │ Prompt │→ │  LLM   │→ answer + citations
 │  validate  │  │ + overlap │  │ (cached) │  │FAISS+BM25  │metadata│  │ BM25+dense+RRF │  │ +rules │  │        │   + logs + cache
 └────────────┘  └───────────┘  └──────────┘  └────────┘   └────────┘  └────────────────┘  └────────┘  └────────┘
                                         ↑ measured with an evaluation set (retrieval + answers) ↑
```

### Table of Contents
1. Setup: imports and one central config
2. The raw documents (a realistic, messy wiki export)
3. Ingestion: validate, de-duplicate, quarantine
4. Chunking with overlap and contextual headers
5. Dense retrieval: embeddings + FAISS (with an embedding cache)
6. Keyword retrieval: BM25
7. Hybrid retrieval with Reciprocal Rank Fusion
8. Metadata filtering: only the latest, active version
9. Evaluating retrieval: hit-rate@k, MRR, stale-document rate
10. Tuning the chunk size with the evaluation set
11. The prompt: answer only from context, cite, or say "I don't know"
12. Generation with an `LLMClient` (local model, optional API)
13. Refusing out-of-scope questions
14. Evaluating answers: citations, groundedness, key facts, refusals
15. Caching, latency and cost logging
16. The final `ask()` function
17. Automated tests
18. Summary, exercises and resources

---
## 1. Setup: imports and one central config

All tunable values (chunk size, overlap, top-k, model names, thresholds) live in **one frozen dataclass**. When someone asks "why did the assistant answer X last Tuesday?", you need to know exactly which settings were live. In a real system this config is versioned in git and written into every log line.

In [1]:
# ---- Standard library ----
import hashlib                    # stable fingerprints for de-duplication and cache keys
import json                       # pretty-printing structured records
import logging                    # status messages with timestamps (instead of print)
import re                         # regular expressions for tokenizing / sentence splitting
import time                       # measuring latency
import warnings
from dataclasses import dataclass, asdict, field
from datetime import date

# ---- Third-party ----
import faiss                      # Meta's fast vector-similarity library (in-memory vector index)
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, ValidationError, field_validator   # data validation
from rank_bm25 import BM25Okapi   # classic keyword search (the algorithm behind Elasticsearch/Lucene scoring)
from sentence_transformers import SentenceTransformer   # turns text into embedding vectors
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()                          # no "Loading weights" progress bars in the output

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)           # hide HTTP chatter from the model hub
log = logging.getLogger("rag")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


@dataclass(frozen=True)
class RAGConfig:
    as_of_date: str = "2026-10-21"          # "today" for the assistant; fixed so the notebook is reproducible
    chunk_size_chars: int = 400             # max characters per chunk (~100 tokens)
    chunk_overlap_chars: int = 80           # characters repeated between neighbouring chunks
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"   # 384-dim, 90 MB, fast on CPU
    top_k: int = 3                          # chunks handed to the LLM
    candidates_per_retriever: int = 20      # how many hits each retriever contributes to fusion
    rrf_k: int = 60                         # constant of Reciprocal Rank Fusion (60 is the usual default)
    min_dense_score: float = 0.25           # below this cosine similarity we refuse without calling the LLM
    min_attribution_score: float = 0.50     # answer-to-source similarity needed to attach a citation ourselves
    max_new_tokens: int = 60                # keep answers short (and the notebook fast on CPU)
    prompt_version: str = "rag-answer-v1"   # bump this whenever the prompt text changes


CFG = RAGConfig()
np.random.seed(0)
print(json.dumps(asdict(CFG), indent=2))

/path/to/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "as_of_date": "2026-10-21",
  "chunk_size_chars": 400,
  "chunk_overlap_chars": 80,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "top_k": 3,
  "candidates_per_retriever": 20,
  "rrf_k": 60,
  "min_dense_score": 0.25,
  "min_attribution_score": 0.5,
  "max_new_tokens": 60,
  "prompt_version": "rag-answer-v1"
}


---
## 2. The raw documents (a realistic, messy wiki export)

In a company, documents come from Confluence, SharePoint, Google Drive, Notion, a CMS or a ticket system, usually through a connector that exports text **plus metadata** (title, owner, version, dates, status).

To keep the notebook offline, we write a small export by hand. It contains the problems you **will** meet in real life:

- **Several versions of the same policy** (remote-work v1 says 2 days at home, v2 says 3 days; travel v2 vs v3 have different limits; two price lists).
- A **draft** that is not approved yet (it promises a feature that does not exist).
- A price list that is **approved but only effective next year** (future-dated).
- An **exact duplicate** (the same page exported twice).
- A document with **missing metadata** (no effective date), which must not reach the index.
- Error codes like `VPN-809` and `SYNC-E1042` that semantic search alone is bad at finding.

In [2]:
RAW_EXPORT = [
    {"doc_id": "HR-REMOTE", "version": 1, "status": "superseded", "effective_date": "2024-03-01",
     "department": "HR", "title": "Remote Work Policy",
     "text": """## Remote days
Employees may work remotely up to 2 days per week. Remote days must be agreed with the line manager in advance.
## Equipment
Lumora pays a one-time home office stipend of $300 for a chair, screen or headset. Receipts are required."""},

    {"doc_id": "HR-REMOTE", "version": 2, "status": "active", "effective_date": "2026-02-01",
     "department": "HR", "title": "Remote Work Policy",
     "text": """## Remote days
Employees may work remotely up to 3 days per week. Core collaboration hours are 10:00 to 15:00 CET, during which everyone must be reachable. Team leads can define one fixed office day per week for their team.
## Working from abroad
Working from another country is allowed for up to 20 working days per calendar year. It requires written approval from HR at least two weeks before travel, because of tax and insurance rules.
## Equipment
Lumora pays a one-time home office stipend of $500 for a chair, screen or headset. Claim it in Expensa with the receipts within 90 days of purchase."""},

    {"doc_id": "FIN-TRAVEL", "version": 2, "status": "superseded", "effective_date": "2023-06-01",
     "department": "Finance", "title": "Travel and Expense Policy",
     "text": """## Meals
The meal per diem is $45 per travel day.
## Hotels
Hotels are reimbursed up to $150 per night in all cities.
## Submitting expenses
Submit expenses within 60 days after the trip."""},

    {"doc_id": "FIN-TRAVEL", "version": 3, "status": "active", "effective_date": "2025-09-01",
     "department": "Finance", "title": "Travel and Expense Policy",
     "text": """## Meals
The meal per diem is $60 per travel day. Alcohol is never reimbursed.
## Hotels
Hotels are reimbursed up to $180 per night. In London, New York and San Francisco the limit is $250 per night. Book through the travel portal TripDesk whenever possible.
## Flights
Book economy class for flights under 6 hours. Premium economy is allowed for flights longer than 6 hours. Business class needs approval from a VP.
## Submitting expenses
Submit all expenses in Expensa within 30 days after the trip. Receipts are required for every item above $25."""},

    {"doc_id": "HR-PTO", "version": 1, "status": "active", "effective_date": "2025-01-01",
     "department": "HR", "title": "Vacation and Sick Leave",
     "text": """## Vacation
Full-time employees receive 28 vacation days per calendar year. Part-time employees receive a pro-rated amount.
## Carry-over
Up to 5 unused vacation days can be carried over to the next year. Carried-over days expire on March 31.
## Sick leave
Report sick leave to your manager before 09:00 on the first day. A doctor's note is required from the third consecutive sick day."""},

    {"doc_id": "HR-PARENTAL", "version": 1, "status": "active", "effective_date": "2025-05-01",
     "department": "HR", "title": "Parental Leave Policy",
     "text": """## Paid leave
The primary caregiver receives 16 weeks of fully paid parental leave. The secondary caregiver receives 6 weeks of fully paid leave.
## Eligibility
Employees are eligible after 6 months of employment. Leave can start up to 4 weeks before the expected birth or adoption date."""},

    {"doc_id": "IT-SECURITY", "version": 4, "status": "active", "effective_date": "2026-01-15",
     "department": "IT", "title": "Information Security Policy",
     "text": """## Passwords and MFA
Passwords must be at least 14 characters long. Multi-factor authentication (MFA) is mandatory for all company accounts.
## Lost or stolen devices
Report a lost or stolen laptop or phone to the IT hotline (extension 4040) within 1 hour. IT will remotely lock and wipe the device.
## Data handling
Customer data must never be copied to USB drives or personal cloud storage. Report suspicious emails with the "Report phishing" button in the mail client."""},

    {"doc_id": "IT-VPN", "version": 1, "status": "active", "effective_date": "2025-11-01",
     "department": "IT", "title": "VPN Troubleshooting Guide",
     "text": """## Error VPN-809
Error VPN-809 means a firewall blocks the connection. Check that outgoing port 443 is open, or switch from hotel Wi-Fi to a mobile hotspot.
## Error VPN-720
Error VPN-720 means your device certificate has expired. Renew it in the self-service portal under Devices > Renew certificate, then restart the VPN client.
## Still not working
Open a ticket in the IT portal and attach the VPN client log file."""},

    {"doc_id": "PROD-PRICING", "version": 1, "status": "superseded", "effective_date": "2025-01-01",
     "department": "Product", "title": "Lumora Sync Price List",
     "text": """## Plans
Starter costs $8 per user per month. Business costs $15 per user per month."""},

    {"doc_id": "PROD-PRICING", "version": 2, "status": "active", "effective_date": "2026-04-01",
     "department": "Product", "title": "Lumora Sync Price List",
     "text": """## Plans
Starter costs $10 per user per month, for up to 10 users and 100 GB of storage per user. Business costs $18 per user per month, with unlimited users, 1 TB of storage per user and single sign-on (SSO). Enterprise pricing is custom; contact the sales team.
## Billing
Annual billing gives a 15% discount compared to monthly billing. Annual plans can be refunded within 30 days of purchase."""},

    {"doc_id": "PROD-PRICING", "version": 3, "status": "active", "effective_date": "2027-01-01",
     "department": "Product", "title": "Lumora Sync Price List",
     "text": """## Plans
From January 2027, Starter costs $12 per user per month and Business costs $21 per user per month."""},

    {"doc_id": "PROD-FAQ", "version": 3, "status": "active", "effective_date": "2026-05-10",
     "department": "Product", "title": "Lumora Sync Product FAQ",
     "text": """## Supported systems
Lumora Sync runs on Windows 11, macOS 14 or newer, and Ubuntu 22.04 or newer. Mobile apps exist for iOS and Android.
## Limits
The maximum size of a single file is 50 GB. Version history is kept for 30 days on Starter and for 180 days on Business.
## Error SYNC-E1042
Sync error SYNC-E1042 means the file path is too long. Windows limits paths to 260 characters; move the file to a shorter folder path.
## Offline mode
Files marked "Available offline" can be edited without internet and are synced automatically when the connection returns."""},

    {"doc_id": "PROD-FAQ", "version": 4, "status": "draft", "effective_date": "2026-11-01",
     "department": "Product", "title": "Lumora Sync Product FAQ",
     "text": """## Limits
DRAFT - NOT APPROVED. Version history will be kept for 365 days on all plans."""},

    {"doc_id": "IT-HARDWARE", "version": 2, "status": "active", "effective_date": "2025-03-01",
     "department": "IT", "title": "Hardware and Equipment Requests",
     "text": """## Laptops
Every employee receives a company laptop. Laptops are replaced every 3 years; request a replacement in the IT portal.
## Monitors and accessories
Office desks have two monitors. For extra accessories (keyboard, mouse, docking station) open a hardware request in the IT portal; items under $150 need no approval.
## Returning equipment
When leaving the company, return all hardware on your last working day."""},

    {"doc_id": "HR-BENEFITS", "version": 1, "status": "active", "effective_date": "2026-01-01",
     "department": "HR", "title": "Employee Benefits Overview",
     "text": """## Learning budget
Each employee has a learning budget of $1,000 per year for courses, books and conferences.
## Wellbeing
Lumora pays up to $40 per month towards a gym or sports membership.
## Public transport
A monthly public transport ticket is reimbursed in full for commuting to the office."""},

    {"doc_id": "FIN-PROCUREMENT", "version": 1, "status": "active", "effective_date": "2025-07-01",
     "department": "Finance", "title": "Purchasing and Software Procurement",
     "text": """## Software
New software tools must be approved by IT security before purchase. Purchases above $5,000 per year need a quote from two vendors.
## Corporate cards
Corporate credit cards may only be used for business expenses. Personal purchases must be repaid within 10 days."""},

    {"doc_id": "PROD-SLA", "version": 2, "status": "active", "effective_date": "2026-03-01",
     "department": "Product", "title": "Lumora Sync Support and SLA",
     "text": """## Uptime
Lumora Sync guarantees 99.9% monthly uptime for Business and Enterprise customers. Customers receive service credits if uptime falls below this level.
## Support response times
Support answers Business customers within 8 business hours and Enterprise customers within 1 hour for critical issues. Starter customers get community and email support."""},

    {"doc_id": "HR-CONDUCT", "version": 3, "status": "active", "effective_date": "2025-10-01",
     "department": "HR", "title": "Code of Conduct",
     "text": """## Gifts
Employees may accept gifts from customers or suppliers worth up to $50. Gifts above this value must be declined or reported to Compliance.
## Speaking up
Concerns can be reported anonymously through the Speak Up hotline."""},

    # The same vacation page exported twice by the wiki connector (exact duplicate)
    {"doc_id": "HR-PTO", "version": 1, "status": "active", "effective_date": "2025-01-01",
     "department": "HR", "title": "Vacation and Sick Leave",
     "text": """## Vacation
Full-time employees receive 28 vacation days per calendar year. Part-time employees receive a pro-rated amount.
## Carry-over
Up to 5 unused vacation days can be carried over to the next year. Carried-over days expire on March 31.
## Sick leave
Report sick leave to your manager before 09:00 on the first day. A doctor's note is required from the third consecutive sick day."""},

    # A page whose metadata is broken (no effective date): it must not reach the index
    {"doc_id": "FIN-BONUS", "version": 1, "status": "active", "effective_date": None,
     "department": "Finance", "title": "Bonus Payout Rules",
     "text": """## Payout
Bonuses are paid in March."""},
]
print(f"{len(RAW_EXPORT)} raw documents exported from the wiki")
pd.DataFrame(RAW_EXPORT).drop(columns="text")

20 raw documents exported from the wiki


,doc_id,version,status,effective_date,department,title
0,HR-REMOTE,1,superseded,2024-03-01,HR,Remote Work Policy
1,HR-REMOTE,2,active,2026-02-01,HR,Remote Work Policy
2,FIN-TRAVEL,2,superseded,2023-06-01,Finance,Travel and Expense Policy
3,FIN-TRAVEL,3,active,2025-09-01,Finance,Travel and Expense Policy
4,HR-PTO,1,active,2025-01-01,HR,Vacation and Sick Leave
5,HR-PARENTAL,1,active,2025-05-01,HR,Parental Leave Policy
6,IT-SECURITY,4,active,2026-01-15,IT,Information Security Policy
7,IT-VPN,1,active,2025-11-01,IT,VPN Troubleshooting Guide
8,PROD-PRICING,1,superseded,2025-01-01,Product,Lumora Sync Price List
9,PROD-PRICING,2,active,2026-04-01,Product,Lumora Sync Price List


---
## 3. Ingestion: validate, de-duplicate, quarantine

Garbage in, garbage out. Before anything is embedded we:

1. **Validate** every record against a schema (a *data contract*). We use **Pydantic**: if a field is missing or has the wrong type, we get a clear error.
2. **Quarantine** invalid records instead of crashing the whole ingestion job (a human fixes them later). Crashing on one bad page would block all good pages; silently indexing it would make the assistant quote a page with unknown validity.
3. **De-duplicate** with a content hash, so the same text is not retrieved three times (that would waste context slots).

In [3]:
class SourceDocument(BaseModel):
    """The data contract every document must satisfy before it may be indexed."""
    doc_id: str = Field(pattern=r"^[A-Z]+-[A-Z]+$")          # e.g. "HR-REMOTE"
    version: int = Field(ge=1)
    status: str
    effective_date: date                                      # Pydantic parses "2026-02-01" into a date
    department: str
    title: str = Field(min_length=3)
    text: str = Field(min_length=20)

    @field_validator("status")
    @classmethod
    def known_status(cls, value: str) -> str:
        allowed = {"active", "superseded", "draft"}
        if value not in allowed:
            raise ValueError(f"status must be one of {allowed}")
        return value

    @property
    def content_hash(self) -> str:
        """Fingerprint of the content: identical text + identity => identical hash."""
        raw = f"{self.doc_id}|{self.version}|{' '.join(self.text.split())}"
        return hashlib.sha256(raw.encode()).hexdigest()[:16]


def ingest(raw_records: list[dict]) -> tuple[list[SourceDocument], list[dict]]:
    """Validate + de-duplicate. Returns (good documents, quarantined records with the reason)."""
    documents, quarantine, seen_hashes = [], [], set()
    for record in raw_records:
        try:
            doc = SourceDocument(**record)
        except ValidationError as err:
            quarantine.append({"doc_id": record.get("doc_id"), "reason": err.errors()[0]["msg"]})
            continue
        if doc.content_hash in seen_hashes:                # exact duplicate of something we already have
            log.info("Skipping duplicate %s v%d", doc.doc_id, doc.version)
            continue
        seen_hashes.add(doc.content_hash)
        documents.append(doc)
    log.info("Ingested %d documents, quarantined %d", len(documents), len(quarantine))
    return documents, quarantine


documents, quarantined = ingest(RAW_EXPORT)
print("Quarantined:", quarantined)

2026-10-21 18:13:42,057 | INFO | Skipping duplicate HR-PTO v1


2026-10-21 18:13:42,059 | INFO | Ingested 18 documents, quarantined 1


Quarantined: [{'doc_id': 'FIN-BONUS', 'reason': 'Input should be a valid date'}]


---
## 4. Chunking with overlap and contextual headers

An LLM prompt should contain only the **few paragraphs that matter**, not whole documents. So we cut documents into **chunks**.

Our strategy (a common industry default):

1. **Split on structure first**: each `## Section` is a natural unit of meaning.
2. **Pack whole sentences** into chunks of at most `chunk_size_chars`. Never cut in the middle of a sentence.
3. **Overlap**: the last sentence(s) of a chunk (up to `chunk_overlap_chars`) are repeated at the start of the next one, so a fact that spans a boundary is not lost.
4. **Contextual header**: prepend `"Title (vN) > Section:"` to the text we index. A chunk that only says *"The limit is $250 per night"* is useless without knowing it is about **hotels** in the **travel policy**. (Anthropic's "Contextual Retrieval" article shows that adding context to chunks cuts retrieval failures substantially; they generate it with an LLM, we use the cheap structural version.)

Every chunk keeps the **metadata** of its document, which we need later for filtering and citations.

In [4]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    version: int
    status: str
    effective_date: date
    department: str
    title: str
    section: str
    text: str                      # the raw chunk text (what we show to the LLM and the user)

    @property
    def indexed_text(self) -> str:
        """Text used for embedding and BM25: chunk + contextual header."""
        return f"{self.title} (v{self.version}) > {self.section}: {self.text}"


def split_sentences(text: str) -> list[str]:
    """Very small sentence splitter: split after . ! ? when followed by a space and a capital/digit."""
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"])", text.strip())
    return [p.strip() for p in parts if p.strip()]


def chunk_section(body: str, size: int, overlap: int) -> list[str]:
    """Pack whole sentences into chunks of <= size characters, repeating ~overlap characters."""
    pieces, current = [], []
    for sentence in split_sentences(body):
        if current and len(" ".join(current + [sentence])) > size:
            pieces.append(" ".join(current))                 # chunk is full: close it
            carry = []                                       # overlap = trailing sentences that fit
            for prev in reversed(current):
                if len(" ".join([prev] + carry)) > overlap:
                    break
                carry.insert(0, prev)
            current = carry
        current.append(sentence)
    if current:
        pieces.append(" ".join(current))
    return pieces


def build_chunks(docs: list[SourceDocument], size: int, overlap: int) -> list[Chunk]:
    out = []
    for doc in docs:
        sections = re.split(r"^## ", doc.text, flags=re.MULTILINE)   # 1) structure-aware split
        for section in filter(str.strip, sections):
            heading, _, body = section.partition("\n")              # first line = section heading
            for i, text in enumerate(chunk_section(body, size, overlap)):   # 2) sentence packing
                out.append(Chunk(
                    chunk_id=f"{doc.doc_id}-v{doc.version}-{heading.strip().lower().replace(' ', '_')}-{i}",
                    doc_id=doc.doc_id, version=doc.version, status=doc.status,
                    effective_date=doc.effective_date, department=doc.department,
                    title=doc.title, section=heading.strip(), text=text))
    return out


chunks = build_chunks(documents, CFG.chunk_size_chars, CFG.chunk_overlap_chars)
lengths = [len(c.text) for c in chunks]
print(f"{len(chunks)} chunks | chars per chunk: min {min(lengths)}, median {int(np.median(lengths))}, max {max(lengths)}")
print("\nExample chunk as it is indexed:\n ", chunks[3].indexed_text)

44 chunks | chars per chunk: min 40, median 119, max 254

Example chunk as it is indexed:
  Remote Work Policy (v2) > Working from abroad: Working from another country is allowed for up to 20 working days per calendar year. It requires written approval from HR at least two weeks before travel, because of tax and insurance rules.


Let's use small settings on one document to see the **overlap** in action: the last sentence of a chunk is repeated at the start of the next one. (We only carry *whole* sentences that fit into the overlap budget; with an 80-character budget this ~90-character sentence would not be carried.)

In [5]:
demo = build_chunks([d for d in documents if d.doc_id == "HR-REMOTE" and d.version == 2], size=160, overlap=120)
for c in demo[:3]:
    print(f"[{c.chunk_id}]\n   {c.text}\n")

[HR-REMOTE-v2-remote_days-0]
   Employees may work remotely up to 3 days per week. Core collaboration hours are 10:00 to 15:00 CET, during which everyone must be reachable.

[HR-REMOTE-v2-remote_days-1]
   Core collaboration hours are 10:00 to 15:00 CET, during which everyone must be reachable. Team leads can define one fixed office day per week for their team.

[HR-REMOTE-v2-working_from_abroad-0]
   Working from another country is allowed for up to 20 working days per calendar year.



---
## 5. Dense retrieval: embeddings + FAISS (with an embedding cache)

An **embedding model** turns a text into a vector (a list of 384 numbers for `all-MiniLM-L6-v2`) such that texts with similar **meaning** get similar vectors. "Can I work from home?" and "remote days per week" share almost no words but end up close together.

**FAISS** stores the vectors and finds the nearest ones quickly. We L2-normalise the vectors and use an inner-product index, which makes the score equal to **cosine similarity** (1 = same direction, 0 = unrelated).

**Embedding cache:** embedding is the most expensive part of ingestion (and costs money with an API). We cache vectors by a hash of `(model name, text)`, so re-running ingestion after one page changed only embeds that page.

In [6]:
embedder = SentenceTransformer(CFG.embedding_model, device="cpu")
EMBEDDING_CACHE: dict[str, np.ndarray] = {}           # in production: a table in Redis/Postgres/the vector DB


def embed(texts: list[str]) -> np.ndarray:
    """Embed texts, re-using cached vectors. Returns an (n, dim) float32 array of unit vectors."""
    keys = [hashlib.sha256(f"{CFG.embedding_model}|{t}".encode()).hexdigest() for t in texts]
    missing = [(k, t) for k, t in zip(keys, texts) if k not in EMBEDDING_CACHE]
    if missing:
        vectors = embedder.encode([t for _, t in missing], normalize_embeddings=True, batch_size=32,
                                  show_progress_bar=False)
        for (k, _), v in zip(missing, vectors):
            EMBEDDING_CACHE[k] = v.astype("float32")
    return np.vstack([EMBEDDING_CACHE[k] for k in keys])


class DenseIndex:
    def __init__(self, chunks: list[Chunk]):
        vectors = embed([c.indexed_text for c in chunks])
        self.index = faiss.IndexFlatIP(vectors.shape[1])   # exact search; fine below ~100k vectors
        self.index.add(vectors)

    def search(self, query: str, k: int, allowed_ids: np.ndarray | None = None) -> list[tuple[int, float]]:
        """Return [(chunk_position, cosine_score)], optionally restricted to allowed chunk positions."""
        q = embed([query])
        params = None
        if allowed_ids is not None:                        # metadata PRE-filter inside FAISS
            params = faiss.SearchParameters(sel=faiss.IDSelectorBatch(allowed_ids.astype("int64")))
            k = min(k, len(allowed_ids))
        scores, ids = self.index.search(q, k, params=params)
        return [(int(i), float(s)) for i, s in zip(ids[0], scores[0]) if i != -1]


t0 = time.perf_counter()
dense = DenseIndex(chunks)
print(f"Indexed {dense.index.ntotal} vectors of dim {dense.index.d} in {time.perf_counter() - t0:.2f}s")
t0 = time.perf_counter()
DenseIndex(chunks)                                          # second build: everything comes from the cache
print(f"Rebuild with warm embedding cache: {time.perf_counter() - t0:.3f}s")

for pos, score in dense.search("Can I work from home?", k=3):
    print(f"  {score:.3f}  {chunks[pos].chunk_id}")

2026-10-21 18:13:42,644 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-10-21 18:13:43,074 | INFO | Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


Indexed 44 vectors of dim 384 in 1.42s
Rebuild with warm embedding cache: 0.001s
  0.467  HR-REMOTE-v2-working_from_abroad-0
  0.354  HR-REMOTE-v1-remote_days-0
  0.330  HR-REMOTE-v2-remote_days-0


Look at the result: both the **current** (v2) and the **outdated** (v1) remote-work policy show up. Semantic search has no idea which one is valid. We fix that in Section 8.

---
## 6. Keyword retrieval: BM25

**BM25** is the classic search-engine scoring function: documents that contain the query's **rare words** many times score high. It has no idea about meaning, but it is excellent at **exact identifiers** such as error codes, product names, SKUs and ticket numbers, where embeddings are often weak. (In our tiny corpus the embedding model also finds `VPN-809`, because the code appears in a short chunk with a matching header; with hundreds of similar codes, embeddings start confusing them.)

In [7]:
STOPWORDS = {"the", "a", "an", "is", "are", "of", "to", "in", "for", "on", "and", "or", "what", "how",
             "do", "does", "i", "my", "me", "can", "be", "it", "with", "at", "by", "this", "that", "much"}


def tokenize(text: str) -> list[str]:
    """Lower-case word tokens; keeps codes like 'vpn-809' and 'sync-e1042' intact."""
    return [t for t in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower()) if t not in STOPWORDS]


class KeywordIndex:
    def __init__(self, chunks: list[Chunk]):
        self.bm25 = BM25Okapi([tokenize(c.indexed_text) for c in chunks])

    def search(self, query: str, k: int, allowed_ids: np.ndarray | None = None) -> list[tuple[int, float]]:
        scores = self.bm25.get_scores(tokenize(query))
        if allowed_ids is not None:                        # same metadata filter as the dense index
            mask = np.full(len(scores), -np.inf)
            mask[allowed_ids] = 0.0
            scores = scores + mask
        order = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in order if scores[i] > 0]


keyword = KeywordIndex(chunks)
query = "VPN-809"
print("BM25 :", [chunks[i].chunk_id for i, _ in keyword.search(query, 3)])
print("Dense:", [chunks[i].chunk_id for i, _ in dense.search(query, 3)])

BM25 : ['IT-VPN-v1-error_vpn-809-0']


Dense: ['IT-VPN-v1-error_vpn-809-0', 'IT-VPN-v1-still_not_working-0', 'IT-VPN-v1-error_vpn-720-0']


---
## 7. Hybrid retrieval with Reciprocal Rank Fusion

Neither retriever is always right, so production systems usually run **both** and merge the results (Elasticsearch, OpenSearch, Weaviate, Qdrant and Vespa all ship "hybrid search").

The scores of BM25 (0 to ~20) and cosine (−1 to 1) are on different scales, so we do not add them. **Reciprocal Rank Fusion (RRF)** only uses the **rank** in each list:

$$\text{RRF}(d) = \sum_{\text{retrievers}} \frac{1}{k + \text{rank}(d)} \qquad (k = 60)$$

A chunk ranked high by both retrievers wins; a chunk found by only one still gets a chance.

In [8]:
def hybrid_search(query: str, k: int, allowed_ids: np.ndarray | None = None,
                  mode: str = "hybrid") -> list[tuple[int, float]]:
    """mode = 'dense' | 'bm25' | 'hybrid'. Returns [(chunk_position, score)] best first."""
    n = CFG.candidates_per_retriever
    if mode == "dense":
        return dense.search(query, k, allowed_ids)
    if mode == "bm25":
        return keyword.search(query, k, allowed_ids)
    fused: dict[int, float] = {}
    for results in (dense.search(query, n, allowed_ids), keyword.search(query, n, allowed_ids)):
        for rank, (pos, _) in enumerate(results, start=1):
            fused[pos] = fused.get(pos, 0.0) + 1.0 / (CFG.rrf_k + rank)
    return sorted(fused.items(), key=lambda item: -item[1])[:k]


for q in ["VPN-809", "My laptop was stolen at the airport"]:
    print(q)
    for mode in ["bm25", "dense", "hybrid"]:
        print(f"   {mode:6s} -> {[chunks[i].chunk_id for i, _ in hybrid_search(q, 2, mode=mode)]}")

VPN-809
   bm25   -> ['IT-VPN-v1-error_vpn-809-0']
   dense  -> ['IT-VPN-v1-error_vpn-809-0', 'IT-VPN-v1-still_not_working-0']
   hybrid -> ['IT-VPN-v1-error_vpn-809-0', 'IT-VPN-v1-still_not_working-0']
My laptop was stolen at the airport
   bm25   -> ['IT-SECURITY-v4-lost_or_stolen_devices-0', 'IT-HARDWARE-v2-laptops-0']


   dense  -> ['IT-SECURITY-v4-lost_or_stolen_devices-0', 'IT-HARDWARE-v2-laptops-0']
   hybrid -> ['IT-SECURITY-v4-lost_or_stolen_devices-0', 'IT-HARDWARE-v2-laptops-0']


---
## 8. Metadata filtering: only the latest, active version

This is the single most important "boring" feature of an enterprise RAG system. Retrieval quality is useless if the assistant confidently quotes **last year's** policy.

Rules (decided with the document owners, not by the ML engineer):
- only `status == "active"` (drafts and superseded versions are excluded),
- only documents whose `effective_date` is **on or before today** (the 2027 price list is not valid yet),
- for each `doc_id`, only the **highest remaining version**.

We apply the filter **before** ranking (pre-filtering). Filtering after taking the top-3 (post-filtering) can leave you with zero results when all top-3 were outdated.

In [9]:
def allowed_chunk_ids(chunks: list[Chunk], as_of: date, department: str | None = None) -> np.ndarray:
    """Positions of chunks that belong to the latest active, already-effective version of each document."""
    meta = pd.DataFrame([{"pos": i, "doc_id": c.doc_id, "version": c.version, "status": c.status,
                          "effective_date": c.effective_date, "department": c.department}
                         for i, c in enumerate(chunks)])
    valid = meta[(meta.status == "active") & (meta.effective_date <= as_of)]
    if department:
        valid = valid[valid.department == department]
    latest = valid.groupby("doc_id")["version"].transform("max")      # highest valid version per doc
    return valid.loc[valid.version == latest, "pos"].to_numpy()


AS_OF = date.fromisoformat(CFG.as_of_date)
ALLOWED = allowed_chunk_ids(chunks, AS_OF)
live_versions = sorted({(chunks[i].doc_id, chunks[i].version) for i in ALLOWED})
print(f"{len(ALLOWED)} of {len(chunks)} chunks are live:", live_versions)

q = "What is the home office equipment stipend?"
print("\nWithout filter:", [chunks[i].chunk_id for i, _ in hybrid_search(q, 3)])
print("With filter   :", [chunks[i].chunk_id for i, _ in hybrid_search(q, 3, ALLOWED)])

36 of 44 chunks are live: [('FIN-PROCUREMENT', 1), ('FIN-TRAVEL', 3), ('HR-BENEFITS', 1), ('HR-CONDUCT', 3), ('HR-PARENTAL', 1), ('HR-PTO', 1), ('HR-REMOTE', 2), ('IT-HARDWARE', 2), ('IT-SECURITY', 4), ('IT-VPN', 1), ('PROD-FAQ', 3), ('PROD-PRICING', 2), ('PROD-SLA', 2)]



Without filter: ['HR-REMOTE-v1-equipment-0', 'HR-REMOTE-v2-equipment-0', 'IT-HARDWARE-v2-monitors_and_accessories-0']
With filter   : ['HR-REMOTE-v2-equipment-0', 'IT-HARDWARE-v2-monitors_and_accessories-0', 'HR-BENEFITS-v1-public_transport-0']


---
## 9. Evaluating retrieval: hit-rate@k, MRR, stale-document rate

"It looks good on three examples" is not an evaluation. We write an **evaluation set**: realistic questions (ideally taken from real help-desk tickets) with the document that contains the answer and a **key fact** the answer must contain. Help-desk staff can write 50 of these in an afternoon; it is the most valuable dataset of the project.

Retrieval metrics:
- **Hit-rate@k**: in what fraction of questions is a chunk from the correct document among the top *k*? (hit@1 = the very first chunk is right.)
- **MRR (Mean Reciprocal Rank)**: average of 1/rank of the first correct chunk (1.0 = always first, 0.5 = typically second).
- **Stale rate@k**: fraction of questions where an outdated/draft/future document appears in the top *k*. Business target: **0**.

The questions are deliberately phrased like employees write: casual, with typos of style, with and without exact keywords.

In [10]:
EVAL_SET = [
    # (question, expected doc_id, key fact that a correct answer must contain)
    ("How many days a week can I work from home?", "HR-REMOTE", "3"),
    ("how many wfh days do we get", "HR-REMOTE", "3"),
    ("What's the home office equipment stipend?", "HR-REMOTE", "500"),
    ("Can I work from Spain for a few weeks?", "HR-REMOTE", "20"),
    ("What is the hotel limit for a business trip to London?", "FIN-TRAVEL", "250"),
    ("per diem for food on business trips?", "FIN-TRAVEL", "60"),
    ("How long do I have to submit my travel expenses?", "FIN-TRAVEL", "30"),
    ("Can I fly premium economy to Singapore (a 12 hour flight)?", "FIN-TRAVEL", "6"),
    ("How many vacation days do I get per year?", "HR-PTO", "28"),
    ("PTO rollover deadline?", "HR-PTO", "March 31"),
    ("When do I need a doctor's note if I'm sick?", "HR-PTO", "third"),
    ("How long is paid parental leave for the primary caregiver?", "HR-PARENTAL", "16"),
    ("My laptop got stolen on the train, what now?", "IT-SECURITY", "4040"),
    ("What is the minimum password length?", "IT-SECURITY", "14"),
    ("I get error VPN-809 when connecting", "IT-VPN", "443"),
    ("The VPN says my certificate expired", "IT-VPN", "self-service portal"),
    ("How much does the Business plan of Lumora Sync cost?", "PROD-PRICING", "18"),
    ("Is there a discount if a customer pays yearly?", "PROD-PRICING", "15%"),
    ("What is the biggest file Lumora Sync can upload?", "PROD-FAQ", "50 GB"),
    ("How far back can Business customers restore old versions of a file?", "PROD-FAQ", "180"),
    ("Customer sees sync error SYNC-E1042", "PROD-FAQ", "260"),
    ("Does Lumora Sync run on Linux?", "PROD-FAQ", "Ubuntu"),
    ("How fast does support reply to Enterprise customers?", "PROD-SLA", "1 hour"),
    ("A supplier wants to give me a $120 bottle of wine, is that ok?", "HR-CONDUCT", "50"),
    # "real user" style: short, indirect, typos, no shared keywords with the document
    ("flying to SF for a conference, how much can the hotel be?", "FIN-TRAVEL", "250"),
    ("can my team lead make me come to the office on mondays", "HR-REMOTE", "one fixed office day"),
    ("how much do we get for udemy courses", "HR-BENEFITS", "1,000"),
    ("customer wants money back on the annual plan after 3 weeks", "PROD-PRICING", "30 days"),
    ("is SSO included in starter?", "PROD-PRICING", "Business"),
    ("lost my phone in a taxi", "IT-SECURITY", "4040"),
    ("vacaton days carryover expirey", "HR-PTO", "March 31"),
]
OUT_OF_SCOPE = [
    "What is Lumora's share price today?",
    "Who will win the next football World Cup?",
    "How many days of bereavement leave do I get?",       # sounds like HR, but no document covers it
    "What is the wifi password of the Berlin office?",
]
LIVE_KEYS = {(chunks[i].doc_id, chunks[i].version) for i in ALLOWED}


def evaluate_retrieval(mode: str, use_filter: bool, k: int = CFG.top_k) -> dict:
    hits, top1, reciprocal_ranks, stale = [], [], [], []
    for question, expected_doc, _ in EVAL_SET:
        results = hybrid_search(question, k, ALLOWED if use_filter else None, mode=mode)
        found = [chunks[i] for i, _ in results]
        ranks = [r for r, c in enumerate(found, start=1) if c.doc_id == expected_doc]
        hits.append(bool(ranks))
        top1.append(bool(ranks) and ranks[0] == 1)
        reciprocal_ranks.append(1.0 / ranks[0] if ranks else 0.0)
        stale.append(any((c.doc_id, c.version) not in LIVE_KEYS for c in found))
    return {"retriever": mode, "metadata filter": use_filter, "hit@1": np.mean(top1), f"hit@{k}": np.mean(hits),
            "MRR": np.mean(reciprocal_ranks), f"stale@{k}": np.mean(stale)}


retrieval_report = pd.DataFrame([evaluate_retrieval(m, f) for f in (False, True) for m in ("bm25", "dense", "hybrid")])
retrieval_report.round(3)

,retriever,metadata filter,hit@1,hit@3,MRR,stale@3
0,bm25,False,0.710,0.871,0.780,0.387
1,dense,False,0.935,1.000,0.968,0.581
2,hybrid,False,0.935,1.000,0.968,0.613
3,bm25,True,0.742,0.806,0.769,0.000
4,dense,True,0.968,1.000,0.984,0.000
5,hybrid,True,0.935,1.000,0.968,0.000


In [11]:
# Error analysis: which questions does each retriever (with the filter) get wrong?
for mode in ["bm25", "dense", "hybrid"]:
    misses = []
    for question, expected_doc, _ in EVAL_SET:
        found = [chunks[i].doc_id for i, _ in hybrid_search(question, CFG.top_k, ALLOWED, mode=mode)]
        if expected_doc not in found:
            misses.append(f"{question!r} -> got {found}")
    print(f"{mode}: {len(misses)} misses")
    for m in misses:
        print("    ", m)

bm25: 6 misses
     'how many wfh days do we get' -> got ['PROD-SLA', 'HR-PTO', 'PROD-FAQ']
     'PTO rollover deadline?' -> got []
     'What is the minimum password length?' -> got []
     'How much does the Business plan of Lumora Sync cost?' -> got ['PROD-SLA', 'PROD-SLA', 'PROD-FAQ']
     'A supplier wants to give me a $120 bottle of wine, is that ok?' -> got []
     'flying to SF for a conference, how much can the hotel be?' -> got ['IT-VPN']
dense: 0 misses
hybrid: 0 misses


**How to read these tables:**
- **Without the metadata filter**, an outdated, draft or future document appears in the top 3 for 39% (BM25) to 61% (hybrid) of the questions (`stale@3`). That is exactly how assistants end up quoting last year's policy. **With the filter, the stale rate is 0 by construction.**
- **BM25 alone** misses 6 of 31 questions with the filter: it has no idea that "wfh" means remote work or that "rollover" means carry-over, and without stemming it does not even match "password" to "Passwords". Its weakness is vocabulary mismatch.
- **Dense retrieval** is the best single retriever here (hit@1 0.968, MRR 0.984), and **hybrid is slightly worse at rank 1** (hit@1 0.935, MRR 0.968): for one question BM25's keyword noise pushed a less relevant chunk to the top. We keep hybrid as the default because in a large corpus with thousands of product codes, ticket numbers and names, exact keyword matching rescues queries that embeddings miss; but this table is the reminder to **decide with your own evaluation set** (a weighted fusion or a re-ranker, see the exercises, can recover that question).
- 🚩 **A perfect hit@3 is a red flag, not a victory.** These questions were written by the same person who wrote the documents, so they share vocabulary. Real employee questions are messier. Before trusting such numbers, sample real tickets into the evaluation set.
- 31 questions is small: one question = ~3 percentage points.

---
## 10. Tuning the chunk size with the evaluation set

Chunk size is a trade-off: small chunks are precise but may lose context; big chunks carry more context but dilute the embedding and cost more prompt tokens. Instead of guessing, re-index with a few settings and **measure**. (The embedding cache makes this cheap: unchanged chunks are not re-embedded.)

In [12]:
def retrieval_for_chunking(size: int, overlap: int) -> dict:
    """Rebuild chunks + indexes with other settings and measure hybrid retrieval with the filter."""
    global chunks, dense, keyword, ALLOWED, LIVE_KEYS
    saved = (chunks, dense, keyword, ALLOWED, LIVE_KEYS)
    chunks = build_chunks(documents, size, overlap)
    dense, keyword = DenseIndex(chunks), KeywordIndex(chunks)
    ALLOWED = allowed_chunk_ids(chunks, AS_OF)
    LIVE_KEYS = {(chunks[i].doc_id, chunks[i].version) for i in ALLOWED}
    row, n_chunks = evaluate_retrieval("hybrid", True), len(chunks)
    context_tokens = np.mean([sum(len(chunks[i].text) for i, _ in hybrid_search(q, CFG.top_k, ALLOWED)) / 4
                              for q, _, _ in EVAL_SET])
    chunks, dense, keyword, ALLOWED, LIVE_KEYS = saved          # restore the production setting
    return {"chunk_size": size, "overlap": overlap, "n_chunks": n_chunks,
            "hit@3": row["hit@3"], "MRR": row["MRR"], "avg context tokens": round(context_tokens)}


pd.DataFrame([retrieval_for_chunking(s, o) for s, o in [(150, 0), (150, 50), (400, 80), (800, 100)]]).round(3)

,chunk_size,overlap,n_chunks,hit@3,MRR,avg context tokens
0,150,0,53,1.0,0.984,79
1,150,50,53,1.0,0.984,80
2,400,80,44,1.0,0.968,104
3,800,100,44,1.0,0.968,104


What the table shows on this corpus:
- **400 and 800 characters give identical chunks**: no section is longer than ~250 characters, so each section becomes exactly one chunk.
- **150 characters** splits sections into single sentences: MRR is a little higher (0.984 vs 0.968, i.e. one question moved from rank 2 to rank 1) and the prompt gets ~24% shorter (79 vs 104 tokens of context per question). Overlap makes no difference here: with a 50-character budget no whole sentence fits, so `(150, 50)` behaves exactly like `(150, 0)`.
- We still keep **400 characters with 80 overlap** (= one complete section per chunk), because a rule and its exceptions then stay together (the hotel limit *and* its city exceptions arrive in the same chunk), which helps the generation step. On this corpus the 150-character setting would be a defensible choice too.

On long PDFs or contracts the differences are usually much larger, which is why this experiment belongs in every RAG project.

---
## 11. The prompt: answer only from context, cite, or say "I don't know"

The prompt is **code**: it is versioned (`prompt_version` in the config), reviewed and tested. Industry-standard rules for a grounded assistant:

1. Answer **only** from the provided sources (no outside knowledge).
2. **Cite** the sources used, with ids the program can check (`[1]`, `[2]`).
3. If the sources don't contain the answer, reply with a **fixed refusal sentence** (a fixed string is easy to detect in code).
4. Keep it short.

Each source block carries its title, version and date, so the model (and the user) can see where a fact comes from.

In [13]:
REFUSAL = "I don't know based on the available documents."

SYSTEM_PROMPT = f"""You are Lumora's internal help-desk assistant.
Rules:
- Answer ONLY with facts from the numbered sources. Do not use outside knowledge.
- After each fact, cite its source number in square brackets, for example [1].
- If the sources do not contain the answer, reply exactly: {REFUSAL}
- Answer in at most 2 short sentences."""


def build_prompt(question: str, retrieved: list[Chunk]) -> str:
    """User message = numbered sources + question. Sources first, question last (works best for small models)."""
    blocks = [f"[{n}] ({c.title}, v{c.version}, effective {c.effective_date}, section: {c.section})\n{c.text}"
              for n, c in enumerate(retrieved, start=1)]
    return "Sources:\n" + "\n\n".join(blocks) + f"\n\nQuestion: {question}\nAnswer:"


example_chunks = [chunks[i] for i, _ in hybrid_search(EVAL_SET[3][0], CFG.top_k, ALLOWED)]
print(SYSTEM_PROMPT, "\n" + "-" * 80)
print(build_prompt(EVAL_SET[3][0], example_chunks))

You are Lumora's internal help-desk assistant.
Rules:
- Answer ONLY with facts from the numbered sources. Do not use outside knowledge.
- After each fact, cite its source number in square brackets, for example [1].
- If the sources do not contain the answer, reply exactly: I don't know based on the available documents.
- Answer in at most 2 short sentences. 
--------------------------------------------------------------------------------
Sources:
[1] (Remote Work Policy, v2, effective 2026-02-01, section: Working from abroad)
Working from another country is allowed for up to 20 working days per calendar year. It requires written approval from HR at least two weeks before travel, because of tax and insurance rules.

[2] (Remote Work Policy, v2, effective 2026-02-01, section: Remote days)
Employees may work remotely up to 3 days per week. Core collaboration hours are 10:00 to 15:00 CET, during which everyone must be reachable. Team leads can define one fixed office day per week for their

### The `LLMClient`: one small wrapper around "whatever model we use"

In a company you almost never call a model SDK directly from 50 places in the code. You write **one thin wrapper** so that:

1. switching provider/model is a **config change**, not a rewrite,
2. every call is **measured the same way** (tokens, latency, cost),
3. retries, timeouts, caching and logging can be added in **one place**.

Our wrapper picks a backend in this order:

| Order | Backend | When |
|---|---|---|
| 1 | Anthropic API (`claude-haiku-5-5` by default, `claude-sonnet-5-5` for harder tasks) | `ANTHROPIC_API_KEY` is set **and** the `anthropic` package is installed |
| 2 | OpenAI API (model name from `OPENAI_MODEL`) | `OPENAI_API_KEY` is set **and** the `openai` package is installed |
| 3 | **Local model** `Qwen/Qwen2.5-0.5B-Instruct` via 🤗 `transformers` on CPU | otherwise (this is what produced the outputs you see) |

> ⚠️ **Honesty note.** The local model has **0.5 billion** parameters. Production models are hundreds of times larger. It will make mistakes that a modern API model would not make. That is fine for this notebook: **the engineering patterns are the point**, and a weak model actually makes it easier to see *why* those patterns (validation, retries, fallbacks, evaluation) exist.

---
## 12. Generation with an `LLMClient`

The code below loads the model once (about 1 GB download the first time, cached afterwards in `~/.cache/huggingface`).

In [14]:
import os
import time
from dataclasses import dataclass, field

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_num_threads(2)          # few threads: faster on a busy shared CPU (raise it on your own machine)
torch.manual_seed(0)              # reproducibility (we also use greedy decoding below)

# Prices in US$ per 1 million tokens (input, output). Check the provider's pricing page before relying on them.
PRICE_PER_MTOK = {
    "claude-sonnet-5-5": (2.00, 10.00),
    "claude-haiku-5-5": (0.10, 0.50),
    "local": (0.0, 0.0),           # runs on our own CPU: no per-token bill (but hardware and ops are not free)
}


@dataclass
class LLMResponse:
    """Everything we want to know about ONE model call (the unit we log, cache and evaluate)."""
    text: str
    model: str
    provider: str
    input_tokens: int
    output_tokens: int
    latency_s: float
    cost_usd: float = 0.0
    cached: bool = False


class LLMClient:
    """Tiny provider-agnostic chat client: Anthropic -> OpenAI -> local transformers model."""

    def __init__(self, local_model: str = "Qwen/Qwen2.5-0.5B-Instruct", prefer: str | None = None,
                 anthropic_model: str = "claude-haiku-5-5", openai_model: str | None = None):
        self.local_model_name = local_model
        self.anthropic_model = anthropic_model
        self.openai_model = openai_model or os.getenv("OPENAI_MODEL", "gpt-5-mini")
        self.provider = prefer or self._detect_provider()
        self._tok = self._model = None                     # local model is loaded lazily (only if needed)
        if self.provider == "local":
            self._load_local()

    @staticmethod
    def _detect_provider() -> str:
        """Use a paid API only if a key AND the SDK are available; otherwise stay fully offline."""
        try:
            if os.getenv("ANTHROPIC_API_KEY"):
                import anthropic  # noqa: F401
                return "anthropic"
        except ImportError:
            pass
        try:
            if os.getenv("OPENAI_API_KEY"):
                import openai  # noqa: F401
                return "openai"
        except ImportError:
            pass
        return "local"

    def _load_local(self):
        t0 = time.perf_counter()
        self._tok = AutoTokenizer.from_pretrained(self.local_model_name)
        self._model = AutoModelForCausalLM.from_pretrained(self.local_model_name, dtype=torch.float32)
        self._model.eval()                                  # inference mode (no dropout)
        print(f"Loaded local model {self.local_model_name} in {time.perf_counter() - t0:.1f}s")

    @property
    def model_name(self) -> str:
        return {"anthropic": self.anthropic_model, "openai": self.openai_model}.get(self.provider, "local")

    def count_tokens(self, text: str) -> int:
        """Token count with the local tokenizer (API providers return exact counts themselves)."""
        if self._tok is None:
            return max(1, len(text) // 4)                   # rough rule of thumb: ~4 characters per token
        return len(self._tok.encode(text))

    def chat(self, system: str, user: str, max_new_tokens: int = 128) -> LLMResponse:
        """One chat turn. temperature=0 / greedy decoding => (almost) deterministic answers."""
        t0 = time.perf_counter()
        if self.provider == "anthropic":
            import anthropic
            msg = anthropic.Anthropic().messages.create(
                model=self.anthropic_model, max_tokens=max_new_tokens, system=system,
                messages=[{"role": "user", "content": user}])
            text = "".join(b.text for b in msg.content if b.type == "text")
            n_in, n_out = msg.usage.input_tokens, msg.usage.output_tokens
        elif self.provider == "openai":
            from openai import OpenAI
            resp = OpenAI().chat.completions.create(
                model=self.openai_model, max_completion_tokens=max_new_tokens,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
            text = resp.choices[0].message.content or ""
            n_in, n_out = resp.usage.prompt_tokens, resp.usage.completion_tokens
        else:
            messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
            prompt = self._tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = self._tok(prompt, return_tensors="pt")
            with torch.no_grad():                           # no gradients needed for inference
                out = self._model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                           pad_token_id=self._tok.eos_token_id)
            new_tokens = out[0, inputs["input_ids"].shape[1]:]          # keep only the generated part
            text = self._tok.decode(new_tokens, skip_special_tokens=True)
            n_in, n_out = int(inputs["input_ids"].shape[1]), int(len(new_tokens))
        price_in, price_out = PRICE_PER_MTOK.get(self.model_name, (0.0, 0.0))
        return LLMResponse(text=text.strip(), model=self.model_name, provider=self.provider,
                           input_tokens=n_in, output_tokens=n_out,
                           latency_s=round(time.perf_counter() - t0, 3),
                           cost_usd=(n_in * price_in + n_out * price_out) / 1e6)


def api_cost_estimate(input_tokens: int, output_tokens: int, model: str = "claude-haiku-5-5") -> float:
    """What the same call WOULD cost on a paid API (useful for budgeting before you switch)."""
    price_in, price_out = PRICE_PER_MTOK[model]
    return (input_tokens * price_in + output_tokens * price_out) / 1e6

In [15]:
llm = LLMClient()
print("Provider:", llm.provider, "| model:", llm.model_name)

question = EVAL_SET[3][0]
response = llm.chat(SYSTEM_PROMPT, build_prompt(question, example_chunks), max_new_tokens=CFG.max_new_tokens)
print(f"\nQ: {question}\nA: {response.text}")
print(f"\n{response.input_tokens} input tokens, {response.output_tokens} output tokens, {response.latency_s:.1f}s")

Loaded local model Qwen/Qwen2.5-0.5B-Instruct in 5.2s
Provider: local | model: local



Q: Can I work from Spain for a few weeks?
A: Yes, you can work from Spain for a few weeks under the remote working policy.

314 input tokens, 18 output tokens, 20.2s


---
## 13. Refusing out-of-scope questions

A help-desk assistant must not guess. We use **two layers** (defence in depth):

1. **Retrieval gate (cheap, deterministic):** if even the best chunk is not similar to the question (cosine below `min_dense_score`), we refuse **without calling the LLM**. This saves money and removes the chance of a hallucinated answer.
2. **Prompt rule:** if some chunks pass the gate but don't actually contain the answer (e.g. "bereavement leave" is similar to the vacation policy), the model is instructed to reply with the refusal sentence.

The threshold is not a guess: we look at the best scores of in-scope vs. out-of-scope questions.

In [16]:
def best_dense_score(question: str) -> float:
    hits = dense.search(question, 1, ALLOWED)
    return hits[0][1] if hits else 0.0


in_scope_scores = [best_dense_score(q) for q, _, _ in EVAL_SET]
out_scope_scores = [best_dense_score(q) for q in OUT_OF_SCOPE]
print(f"In-scope questions : best cosine min {min(in_scope_scores):.2f} | median {np.median(in_scope_scores):.2f}")
for q, s in zip(OUT_OF_SCOPE, out_scope_scores):
    print(f"Out-of-scope {s:.2f} : {q}")
print(f"\nGate threshold = {CFG.min_dense_score}: blocks {sum(s < CFG.min_dense_score for s in out_scope_scores)}"
      f"/{len(OUT_OF_SCOPE)} out-of-scope and {sum(s < CFG.min_dense_score for s in in_scope_scores)}"
      f"/{len(EVAL_SET)} in-scope questions")

In-scope questions : best cosine min 0.28 | median 0.59
Out-of-scope 0.57 : What is Lumora's share price today?
Out-of-scope 0.10 : Who will win the next football World Cup?
Out-of-scope 0.49 : How many days of bereavement leave do I get?
Out-of-scope 0.34 : What is the wifi password of the Berlin office?

Gate threshold = 0.25: blocks 1/4 out-of-scope and 0/31 in-scope questions


We set the threshold to **0.25**, just below the lowest score of a real question (0.28), so that no legitimate question is blocked. With that setting the gate only stops the clearly unrelated World Cup question. Questions that *sound* like company topics get scores in the same range as real questions: "Lumora's share price" scores 0.57 simply because it contains the company name, and "bereavement leave" lands close to the vacation policy. They pass the gate, and **the LLM's refusal rule has to do the work**. That is the realistic situation: no single similarity threshold separates in-scope from out-of-scope questions, which is why we need both layers and an evaluation that measures them.

---
## 14. Evaluating answers: citations, groundedness, key facts, refusals

Now we run the **whole pipeline** on the evaluation set and check each answer automatically. These are cheap, deterministic checks in the spirit of Hamel Husain's "Level 1" evals (unit-test-like assertions); companies later add LLM-as-judge and human review on top.

| Check | How | Why |
|---|---|---|
| **Key fact** | the expected fact string (e.g. `250`) appears in the answer | correctness |
| **Model cited** | the model itself wrote a valid `[n]` | does the model follow the format? |
| **Has citation** | after *attribution* (below) the answer points to a source | traceability |
| **Correct source cited** | the cited source belongs to the expected document | the user can verify the answer |
| **Numbers grounded** | every number in the answer also appears in the retrieved sources | catches invented limits/prices (classic hallucination) |
| **Refusal** | out-of-scope answer equals the refusal sentence | "I don't know" beats a wrong answer |

**Post-hoc citation attribution.** Models, small ones especially, often forget the `[n]` format or cite the wrong number. Many production systems therefore do not rely on the model alone: when the model gives no valid citation, the system compares the answer with each retrieved source (embedding similarity) and attaches the most similar one, but only if it is similar enough (`min_attribution_score`). Either way, the **checks** decide whether the citation is correct.

The full pipeline function (`answer_question`) is also the core of the final `ask()` in Section 16.

In [17]:
CALL_LOG: list[dict] = []          # one record per question: our observability data (in production: a log/trace store)


def attribute_citations(answer: str, retrieved: list[Chunk]) -> tuple[list[int], str]:
    """Return (citation numbers, how). Trust valid model citations; otherwise attach the most similar source."""
    model_cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer) if 1 <= int(n) <= len(retrieved)})
    if model_cited:
        return model_cited, "model"
    text = re.sub(r"\[\d+\]", "", answer).strip()
    if not text or not retrieved:
        return [], "none"
    similarity = embed([text]) @ embed([c.text for c in retrieved]).T          # cosine (unit vectors)
    best = int(similarity[0].argmax())
    if similarity[0, best] >= CFG.min_attribution_score:
        return [best + 1], "attributed"
    return [], "none"


def answer_question(question: str, top_k: int = CFG.top_k) -> dict:
    """Retrieve -> gate -> prompt -> LLM -> parse citations. Logs latency and token usage."""
    t0 = time.perf_counter()
    results = hybrid_search(question, top_k, ALLOWED)
    retrieved = [chunks[i] for i, _ in results]
    retrieval_ms = (time.perf_counter() - t0) * 1000
    record = {"question": question, "retrieval_ms": round(retrieval_ms, 1), "generation_s": 0.0,
              "input_tokens": 0, "output_tokens": 0, "sources": [c.chunk_id for c in retrieved]}

    if best_dense_score(question) < CFG.min_dense_score:            # layer 1: retrieval gate
        answer, record["gated"] = REFUSAL, True
    else:
        response = llm.chat(SYSTEM_PROMPT, build_prompt(question, retrieved), max_new_tokens=CFG.max_new_tokens)
        answer, record["gated"] = response.text, False
        record.update(generation_s=response.latency_s, input_tokens=response.input_tokens,
                      output_tokens=response.output_tokens)
    model_citations = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)})
    cited_numbers, how = attribute_citations(answer, retrieved) if not record["gated"] else ([], "none")
    record.update(answer=answer, model_citations=model_citations, cited=cited_numbers, citation_how=how,
                  retrieved=retrieved,
                  total_s=round(time.perf_counter() - t0, 2))
    CALL_LOG.append({k: v for k, v in record.items() if k not in ("retrieved", "model_citations")})
    return record


def numbers_in(text: str) -> set[str]:
    """All numbers in a text, normalised ("$1,000" -> "1000") so formatting differences don't matter."""
    return set(re.findall(r"\d+(?:\.\d+)?", text.replace(",", "")))


def check_answer(record: dict, expected_doc: str | None, key_fact: str | None) -> dict:
    answer, retrieved = record["answer"], record["retrieved"]
    is_refusal = "i don't know" in answer.lower()
    cited = record["cited"]
    context_text = " ".join(c.indexed_text for c in retrieved)
    answer_wo_citations = re.sub(r"\[\d+\]", "", answer)
    return {
        "refused": is_refusal,
        "key_fact": (key_fact.lower().replace(",", "") in answer.lower().replace(",", "")) if key_fact else None,
        "model_cited": bool(record["model_citations"]) and all(1 <= n <= len(retrieved) for n in record["model_citations"]),
        "valid_citation": bool(cited) and all(1 <= n <= len(retrieved) for n in cited),
        "correct_source_cited": any(1 <= n <= len(retrieved) and retrieved[n - 1].doc_id == expected_doc
                                    for n in cited) if expected_doc else None,
        "numbers_grounded": numbers_in(answer_wo_citations) <= numbers_in(context_text),
        "cites_stale_doc": any(1 <= n <= len(retrieved) and
                               (retrieved[n - 1].doc_id, retrieved[n - 1].version) not in LIVE_KEYS for n in cited),
    }


# Answer-level evaluation costs one LLM call per question, so we run it on a fixed, varied sample
# (exact codes, paraphrases, typos, conflicting versions). In CI you would run the full set nightly.
ANSWER_SAMPLE = [EVAL_SET[i] for i in (1, 3, 5, 12, 14, 16, 19, 23, 24, 28)]

t0 = time.perf_counter()
rows = []
for question, expected_doc, key_fact in ANSWER_SAMPLE:
    rec = answer_question(question)
    rows.append({"question": question, "answer": rec["answer"], "citation": rec["citation_how"],
                 **check_answer(rec, expected_doc, key_fact)})
answers_df = pd.DataFrame(rows)
print(f"Answered {len(ANSWER_SAMPLE)} questions in {time.perf_counter() - t0:.0f}s")
answers_df[["question", "answer", "citation"]]

Answered 10 questions in 110s


,question,answer,citation
0,how many wfh days do we get,We have 4 WFLH days.,none
1,Can I work from Spain for a few weeks?,"Yes, you can work from Spain for a few weeks under the remote working policy.",attributed
2,per diem for food on business trips?,"Based on the information provided, the meal per diem for business trips is $60 per travel day.",attributed
3,"My laptop got stolen on the train, what now?",IT will remotely lock and wipe the laptop.,attributed
4,I get error VPN-809 when connecting,Check that outgoing port 443 is open and try switching from hotel Wi-Fi to a mobile hotspot.,attributed
5,How much does the Business plan of Lumora Sync cost?,"The Business plan of Lumora Sync costs $18 per user per month, with unlimited users, 1 TB of storage per user and si...",attributed
6,How far back can Business customers restore old versions of a file?,Business customers can restore old versions of a file up to 180 days before their expiration date.,attributed
7,"A supplier wants to give me a $120 bottle of wine, is that ok?","Yes, it is acceptable as long as the total cost does not exceed $50.",none
8,"flying to SF for a conference, how much can the hotel be?","Based on the information provided, the hotel can cover meals and alcohol costs of $60 per travel day. For a conferen...",attributed
9,is SSO included in starter?,"Yes, SSO is included in the Starter plan.",none


In [18]:
oos_rows = []
for question in OUT_OF_SCOPE:
    rec = answer_question(question)
    oos_rows.append({"question": question, "answer": rec["answer"], "gated": rec["gated"],
                     **check_answer(rec, None, None)})
oos_df = pd.DataFrame(oos_rows)

scorecard = pd.Series({
    "key fact correct (in-scope)": answers_df["key_fact"].mean(),
    "model wrote a valid [n] citation (in-scope)": answers_df["model_cited"].mean(),
    "has a citation after attribution (in-scope)": answers_df["valid_citation"].mean(),
    "correct source cited (in-scope)": answers_df["correct_source_cited"].mean(),
    "all numbers grounded (in-scope)": answers_df["numbers_grounded"].mean(),
    "wrongly refused (in-scope)": answers_df["refused"].mean(),
    "answers citing a stale document": answers_df["cites_stale_doc"].sum(),
    "refused (out-of-scope)": oos_df["refused"].mean(),
}).round(3)
print(scorecard.to_string())
oos_df[["question", "gated", "answer"]]

key fact correct (in-scope)                    0.50
model wrote a valid [n] citation (in-scope)    0.00
has a citation after attribution (in-scope)    0.70
correct source cited (in-scope)                0.60
all numbers grounded (in-scope)                0.80
wrongly refused (in-scope)                     0.00
answers citing a stale document                0.00
refused (out-of-scope)                         0.75


,question,gated,answer
0,What is Lumora's share price today?,False,"Based on the information provided, Lumora's share price today is not mentioned in the given sources."
1,Who will win the next football World Cup?,True,I don't know based on the available documents.
2,How many days of bereavement leave do I get?,False,I don't know based on the available documents.
3,What is the wifi password of the Berlin office?,False,I don't know based on the available documents.


In [19]:
# Error analysis: look at every failing in-scope answer (this is where you learn the most)
failures = answers_df[~answers_df["key_fact"] | ~answers_df["correct_source_cited"] | ~answers_df["numbers_grounded"]]
for _, row in failures.iterrows():
    flags = [name for name in ["key_fact", "correct_source_cited", "numbers_grounded"] if not row[name]]
    print(f"- {row['question']}\n    answer: {row['answer']}\n    failed: {flags}")

- how many wfh days do we get
    answer: We have 4 WFLH days.
    failed: ['key_fact', 'correct_source_cited', 'numbers_grounded']
- Can I work from Spain for a few weeks?
    answer: Yes, you can work from Spain for a few weeks under the remote working policy.
    failed: ['key_fact']
- My laptop got stolen on the train, what now?
    answer: IT will remotely lock and wipe the laptop.
    failed: ['key_fact']
- How much does the Business plan of Lumora Sync cost?
    answer: The Business plan of Lumora Sync costs $18 per user per month, with unlimited users, 1 TB of storage per user and single sign-on (SSO).
    failed: ['correct_source_cited']
- A supplier wants to give me a $120 bottle of wine, is that ok?
    answer: Yes, it is acceptable as long as the total cost does not exceed $50.
    failed: ['correct_source_cited']
- flying to SF for a conference, how much can the hotel be?
    answer: Based on the information provided, the hotel can cover meals and alcohol costs of $60 per 

**Reading the scorecard honestly** (10 in-scope answers, 4 out-of-scope questions):

| KPI from the business problem | Target | This run | Met? |
|---|---|---|---|
| Retrieval hit-rate@3 | ≥ 90% | 100% (hybrid, filtered) | yes (with the red-flag caveat above) |
| Answers citing an outdated/draft document | 0 | 0 | yes, guaranteed by the metadata filter |
| Out-of-scope questions refused | ≥ 90% | 75% (3 of 4) | **no** |
| Answers with a valid citation | ≥ 90% | 70% (after attribution) | **no** |

What the failure list shows:
- **The model never wrote a `[n]` citation by itself** (0 of 10). Without post-hoc attribution no answer would be traceable; with it, 7 of 10 get a citation, and 6 of those point to the right document.
- **Numeric grounding catches invented numbers**: "4 WFLH days" (the policy says 3) and "$360 per day" for San Francisco (the model multiplied numbers on its own) contain numbers that are in no source, so `numbers_grounded` is False. In `ask()` such answers are replaced by the extractive fallback.
- **Some wrong answers pass the cheap checks.** "Yes, SSO is included in the Starter plan" is wrong (SSO is a Business feature) but contains no number. The wine answer contains the key fact "$50" but draws the wrong conclusion for a $120 gift. Substring and number checks are a good first layer; the next layers are an **LLM-as-judge** (a strong model grading faithfulness, as in RAGAS) and **human review** of a sample.
- **Out-of-scope:** the gate stopped only the World Cup question (no LLM call, no cost). For the share price the model answered "...is not mentioned in the given sources": in spirit a refusal, but not the exact sentence, so our strict check counts it as a failure. That is why the prompt asks for a fixed refusal string: anything else is hard to detect reliably in code.

**What companies do next with such failures:** use a stronger model (a production model follows citation and refusal instructions far more reliably: **re-run this exact scorecard after switching the backend**), add few-shot examples of citing and refusing, keep the post-checks with the *extractive fallback* (Section 16), and add every real failure to the evaluation set so it can never silently come back.

---
## 15. Caching, latency and cost logging

Every answer above was logged in `CALL_LOG`. That log is what you look at when someone says "the assistant is slow" or "the bill doubled". Typical dashboards show **p50 / p95 latency**, tokens per question and cost per 1,000 questions.

Our local model costs $0 per token, so we also estimate what the same traffic would cost on a paid API (`claude-haiku-5-5`, prices in the client cell).

In [20]:
log_df = pd.DataFrame(CALL_LOG)
llm_calls = log_df[~log_df["gated"]]
print(f"Questions logged: {len(log_df)} | LLM calls: {len(llm_calls)} | gated (no LLM call): {log_df['gated'].sum()}")
print(f"Retrieval latency  p50 {log_df['retrieval_ms'].quantile(.5):6.1f} ms | p95 {log_df['retrieval_ms'].quantile(.95):6.1f} ms")
print(f"Generation latency p50 {llm_calls['generation_s'].quantile(.5):6.2f} s  | p95 {llm_calls['generation_s'].quantile(.95):6.2f} s (CPU, 0.5B model)")
print(f"Tokens per LLM call: {llm_calls['input_tokens'].mean():.0f} in / {llm_calls['output_tokens'].mean():.0f} out")

monthly_questions = 2000
cost_per_q = api_cost_estimate(llm_calls["input_tokens"].mean(), llm_calls["output_tokens"].mean(), "claude-haiku-5-5")
cost_per_q_sonnet = api_cost_estimate(llm_calls["input_tokens"].mean(), llm_calls["output_tokens"].mean(), "claude-sonnet-5-5")
print(f"\nEstimated API cost: ${cost_per_q * 1000:.3f} per 1,000 questions on claude-haiku-5-5, "
      f"${cost_per_q_sonnet * 1000:.2f} on claude-sonnet-5-5")
print(f"Help-desk cost avoided if 50% of {monthly_questions} monthly questions are answered: ${monthly_questions * 0.5 * 6:,.0f}/month")

Questions logged: 14 | LLM calls: 13 | gated (no LLM call): 1
Retrieval latency  p50    0.6 ms | p95    1.0 ms
Generation latency p50   7.80 s  | p95  17.93 s (CPU, 0.5B model)
Tokens per LLM call: 305 in / 20 out

Estimated API cost: $0.040 per 1,000 questions on claude-haiku-5-5, $0.81 on claude-sonnet-5-5
Help-desk cost avoided if 50% of 2000 monthly questions are answered: $6,000/month


**Caching.** Many employees ask the same thing ("how many vacation days?"). An **answer cache** returns a stored answer in microseconds. The cache key must contain everything that can change the answer: the normalised question, the prompt version, the **index version** (a hash of the live documents) and the model. When a policy changes, the index version changes and old answers are automatically ignored, so the cache can never serve a stale policy.

(Providers also offer **prompt caching** of long, repeated prompt prefixes, which cuts cost for the system prompt; that is complementary to an answer cache.)

In [21]:
INDEX_VERSION = hashlib.sha256("|".join(sorted(f"{chunks[i].doc_id}:{chunks[i].version}" for i in ALLOWED))
                               .encode()).hexdigest()[:12]
ANSWER_CACHE: dict[str, dict] = {}          # in production: Redis with a time-to-live (TTL)


def cache_key(question: str) -> str:
    normalised = " ".join(re.findall(r"[a-z0-9%$]+", question.lower()))     # case/punctuation-insensitive
    return hashlib.sha256(f"{normalised}|{CFG.prompt_version}|{INDEX_VERSION}|{llm.model_name}".encode()).hexdigest()


print("Index version:", INDEX_VERSION)
print(cache_key("How many vacation days do I get?") == cache_key("how many vacation days do I get"))

Index version: 199e40dbd390
True


---
## 16. The final `ask()` function

This is what the chat UI, Slack bot or API endpoint (e.g. FastAPI, see `12_FastAPI/`) would call. It combines everything:

1. **Input validation** with Pydantic (empty or 5,000-character inputs are rejected).
2. **Cache** lookup.
3. Retrieval with the **metadata filter** and **hybrid search**.
4. **Gate** + LLM generation with the grounded prompt.
5. **Citations + post-checks**: model citations or post-hoc attribution; if the answer still has no citation, or contains numbers that are not in the sources, we fall back to an **extractive answer** (the best source passage, clearly labelled). A slightly clunky answer is better than a confident wrong one.
6. A structured result with **citations** (title, version, date) that the UI can display as links.

In [22]:
class AskRequest(BaseModel):
    question: str = Field(min_length=3, max_length=500)
    department: str | None = None                       # optional filter, e.g. "IT" when asked from the IT portal


class Citation(BaseModel):
    number: int
    title: str
    version: int
    effective_date: date
    section: str


class AskResult(BaseModel):
    answer: str
    citations: list[Citation]
    status: str                                          # "answered" | "extractive_fallback" | "refused"
    cached: bool
    latency_s: float


def ask(question: str, department: str | None = None) -> AskResult:
    t0 = time.perf_counter()
    request = AskRequest(question=question, department=department)     # raises ValidationError on bad input
    key = cache_key(request.question + f"|{request.department}")
    if key in ANSWER_CACHE:
        return AskResult(**{**ANSWER_CACHE[key], "cached": True, "latency_s": round(time.perf_counter() - t0, 4)})

    allowed = allowed_chunk_ids(chunks, AS_OF, request.department) if request.department else ALLOWED
    retrieved = [chunks[i] for i, _ in hybrid_search(request.question, CFG.top_k, allowed)]
    best = dense.search(request.question, 1, allowed)
    if not retrieved or not best or best[0][1] < CFG.min_dense_score:
        answer, status, cited = REFUSAL, "refused", []
    else:
        answer = llm.chat(SYSTEM_PROMPT, build_prompt(request.question, retrieved), CFG.max_new_tokens).text
        cited, _ = attribute_citations(answer, retrieved)               # model citation, or attributed by us
        grounded = numbers_in(re.sub(r"\[\d+\]", "", answer)) <= numbers_in(" ".join(c.indexed_text for c in retrieved))
        if "i don't know" in answer.lower():
            answer, status, cited = REFUSAL, "refused", []
        elif not cited or not grounded:                  # post-check failed -> safe extractive fallback
            answer, status, cited = f"From the policy: \"{retrieved[0].text}\" [1]", "extractive_fallback", [1]
        else:
            status = "answered"
    citations = [Citation(number=n, title=retrieved[n - 1].title, version=retrieved[n - 1].version,
                          effective_date=retrieved[n - 1].effective_date, section=retrieved[n - 1].section)
                 for n in cited]
    result = {"answer": answer, "citations": [c.model_dump() for c in citations], "status": status}
    ANSWER_CACHE[key] = result
    return AskResult(**result, cached=False, latency_s=round(time.perf_counter() - t0, 3))


for q in ["What's the hotel limit in New York?", "How many days of bereavement leave do I get?",
          "What's the hotel limit in New York?"]:                     # the 3rd call is a cache hit
    r = ask(q)
    print(f"Q: {q}\n   [{r.status}, cached={r.cached}, {r.latency_s}s] {r.answer}")
    for c in r.citations:
        print(f"   [{c.number}] {c.title} v{c.version} (effective {c.effective_date}) > {c.section}")

Q: What's the hotel limit in New York?
   [answered, cached=False, 8.165s] The hotel limit in New York is $250 per night.
   [1] Travel and Expense Policy v3 (effective 2025-09-01) > Hotels


Q: How many days of bereavement leave do I get?
   [refused, cached=False, 5.655s] I don't know based on the available documents.
Q: What's the hotel limit in New York?
   [answered, cached=True, 0.0s] The hotel limit in New York is $250 per night.
   [1] Travel and Expense Policy v3 (effective 2025-09-01) > Hotels


In [23]:
# The department filter: the same question asked from the IT portal only searches IT documents
r = ask("What should I do if my certificate expired?", department="IT")
print(r.status, "|", r.answer, "|", [(c.title, c.version) for c in r.citations])

# Bad input is rejected with a clear message instead of reaching the model
try:
    ask("")
except ValidationError as err:
    print("Rejected:", err.errors()[0]["msg"])

answered | Renew your certificate by visiting the self-service portal under Devices > Renew certificate, then restarting the VPN client. | [('VPN Troubleshooting Guide', 1)]
Rejected: String should have at least 3 characters


---
## 17. Automated tests

These run in CI on every change to the prompt, the documents, the chunker or the model (see `15_MLOps/07_testing_in_ml/`). The quality gate uses the **evaluation set**: a pull request that drops hit-rate@3 below the target is blocked.

In [24]:
def test_no_stale_documents_are_retrievable():
    for question, _, _ in EVAL_SET:
        for pos, _ in hybrid_search(question, 10, ALLOWED):
            assert chunks[pos].status == "active" and chunks[pos].effective_date <= AS_OF


def test_latest_version_wins():
    hit = chunks[hybrid_search("home office equipment stipend", 1, ALLOWED)[0][0]]
    assert (hit.doc_id, hit.version) == ("HR-REMOTE", 2), hit.chunk_id


def test_invalid_documents_are_quarantined():
    assert any(q["doc_id"] == "FIN-BONUS" for q in quarantined)
    assert not any(c.doc_id == "FIN-BONUS" for c in chunks)


def test_retrieval_quality_gate():
    assert evaluate_retrieval("hybrid", True)["hit@3"] >= 0.90


def test_unrelated_question_is_refused_without_llm_call():
    assert ask("Who will win the next football World Cup?").status == "refused"


def test_cache_returns_same_answer():
    first, second = ask("How many vacation days do I get per year?"), ask("how many vacation days do I get per year")
    assert second.cached and first.answer == second.answer


for test in [test_no_stale_documents_are_retrievable, test_latest_version_wins, test_invalid_documents_are_quarantined,
             test_retrieval_quality_gate, test_unrelated_question_is_refused_without_llm_call,
             test_cache_returns_same_answer]:
    test()
    print(f"PASSED  {test.__name__}")

PASSED  test_no_stale_documents_are_retrievable
PASSED  test_latest_version_wins
PASSED  test_invalid_documents_are_quarantined
PASSED  test_retrieval_quality_gate
PASSED  test_unrelated_question_is_refused_without_llm_call


PASSED  test_cache_returns_same_answer


---
## 18. Summary, exercises and resources

| Practice | What we did | Where it is used in industry |
|---|---|---|
| Data contract at ingestion | Pydantic `SourceDocument`, quarantine, content-hash de-dup | Every connector-based RAG (Confluence, SharePoint, Drive) |
| Structure-aware chunking + overlap | Split by section, pack sentences, overlap, contextual header | LangChain/LlamaIndex splitters, Anthropic "Contextual Retrieval" |
| Embedding cache | Hash of (model, text) → vector | Incremental re-indexing, lower embedding bills |
| Hybrid search | BM25 + dense + Reciprocal Rank Fusion | Elasticsearch/OpenSearch, Weaviate, Qdrant, Vespa |
| Metadata pre-filtering | Latest active, effective version; department filter | Policy/legal/support assistants, multi-tenant search |
| Retrieval evaluation | hit@k, MRR, stale@k, chunk-size experiment | Before every index/model change |
| Grounded prompt | Sources with ids, citations, fixed refusal sentence | Customer-support and internal-knowledge bots |
| Two-layer refusal | Similarity gate + prompt rule | Saves cost, reduces hallucinations |
| Answer checks | key fact, citation validity, numeric grounding, refusals | "Level 1" evals, RAGAS-style faithfulness |
| Safe fallback | Extractive answer when post-checks fail | Regulated domains (HR, finance, health) |
| Observability | Per-call log, p50/p95 latency, tokens, cost per 1k questions | LangSmith, Langfuse, Arize Phoenix, OpenTelemetry |
| Versioned cache | key = question + prompt version + index version + model | Redis/semantic caches in front of LLMs |

### Practice exercises
1. Add a new document version `FIN-TRAVEL` v4 (per diem $70, effective last month). Re-run the notebook: which tests and eval rows change? Did the cache serve any stale answer?
2. Add 10 more evaluation questions from your own imagination (including two with typos). How do hit@3 and MRR change? Which retriever suffers most from typos?
3. Add a **cross-encoder re-ranker** (`cross-encoder/ms-marco-MiniLM-L6-v2`, see `02_rag/04_advanced_retrieval_and_reranking.ipynb`): retrieve 20 hybrid candidates, re-rank, keep 3. Does MRR improve?
4. Write a few-shot version of the prompt (`rag-answer-v2`) with one example of citing and one of refusing. Compare its scorecard with v1 (the method is in `20_industry_llm_application_patterns.ipynb`).
5. If you have an API key, set `ANTHROPIC_API_KEY`, re-run, and compare the scorecard of `claude-haiku-5-5` with the local 0.5B model.

### Resources
- Anthropic, *Introducing Contextual Retrieval* (BM25 + embeddings + context headers, top-k study): https://www.anthropic.com/news/contextual-retrieval
- Anthropic, *Building effective agents* (start with the simplest solution, e.g. one LLM call with retrieval): https://www.anthropic.com/research/building-effective-agents
- Claude cookbooks, `capabilities/retrieval_augmented_generation`: https://github.com/anthropics/claude-cookbooks
- OpenAI Cookbook (RAG and evaluation examples): https://github.com/openai/openai-cookbook
- RAGAS metrics (faithfulness, context precision/recall): https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/
- Hamel Husain, *Your AI Product Needs Evals*: https://hamel.dev/blog/posts/evals/
- Chip Huyen, *Building LLM applications for production*: https://huyenchip.com/2023/04/11/llm-engineering.html
- Eugene Yan, *Patterns for Building LLM-based Systems & Products*: https://eugeneyan.com/writing/llm-patterns/
- Libraries used: FAISS https://github.com/facebookresearch/faiss · rank_bm25 https://github.com/dorianbrown/rank_bm25 · Sentence-Transformers https://sbert.net · Qwen2.5 https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct

### Related notebooks in this repo
- RAG building blocks: `02_rag/01_rag_fundamentals.ipynb`, `02_rag/02_advanced_rag.ipynb`, `02_rag/04_advanced_retrieval_and_reranking.ipynb`
- Frameworks (LangChain, LlamaIndex, Haystack): `02_rag/03_rag_frameworks.ipynb`
- Evaluation theory: `08_evaluation/01_llm_evaluation.ipynb`
- Security (prompt injection through documents): `16_llm_security.ipynb`
- The engineering around LLM calls (prompt versioning, retries, guardrails, eval harness): `20_industry_llm_application_patterns.ipynb`
- Agents that use tools: `../10_Agents/09_industry_tool_using_agent.ipynb`